In [1]:
import os
import numpy as np
import time
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# --- Step 0: Start timer ---
start_time = time.time()

# --- Step 1: Set paths ---
home = os.path.expanduser('~')
file_path = os.path.join(home, 'Downloads', 'Loughran-McDonald_10X_DocumentDictionaries_1993-2024.txt')

# --- Constants ---
target_form = '10-K'
vocab_size = 100000
current_year = 2024
earliest_year = current_year - 20  # Only keep 2004 and newer

# --- Step 2: Load S&P 500 CIKs and Symbols ---

sp500_df = pd.read_csv('inputs/sp500.csv')

sp500_df['CIK'] = sp500_df['CIK'].astype(str)  # Remove leading zeros and capitalize
sp500_ciks = sp500_df['CIK'].tolist()

# Make a lookup dictionary: CIK → Symbol
cik_to_symbol = dict(zip(sp500_df['CIK'], sp500_df['Symbol']))

# --- Functions ---
def parse_word_counts(wordcount_part):
    counts = {}
    for pair in wordcount_part.strip().split(','):
        if ':' in pair:
            try:
                seq, count = map(int, pair.split(':'))
                counts[seq] = count
            except ValueError:
                continue
    return counts

def filing_to_vector(word_counts, vocab_size):
    vector = np.zeros(vocab_size)
    for seq, count in word_counts.items():
        if seq < vocab_size:
            vector[seq] = count
    return vector

# --- Step 3: Read 10-K data and collect filings ---
all_filings = {}  # {CIK: [(period_end, filing_date, word_counts), ...]}

with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            header_part, wordcount_part = line.strip().split('|', 1)
            fields = header_part.split(',')
            CIK = fields[0].lstrip('0').upper()
            filing_date = fields[1]  # Get the filing date
            form_type = fields[4]
            if CIK in sp500_ciks and form_type == target_form:
                period_end = int(fields[3])
                word_counts = parse_word_counts(wordcount_part)
                if CIK not in all_filings:
                    all_filings[CIK] = []
                all_filings[CIK].append((period_end, filing_date, word_counts))
        except Exception:
            continue

# --- Step 4: Process filings and calculate cosine distances ---
final_results = []

for CIK in sp500_ciks:
    if CIK not in all_filings:
        continue

    filings = all_filings[CIK]
    filings.sort(key=lambda x: x[0])  # sort by period_end

    previous_vector = None
    previous_filing_date = None

    for period_end, filing_date, word_counts in filings:
        year = int(str(period_end)[:4])
        current_vector = filing_to_vector(word_counts, vocab_size)

        if previous_vector is not None:
            cos_sim = cosine_similarity([previous_vector], [current_vector])[0][0]
            cos_distance = 1 - cos_sim

            if year >= earliest_year:
                final_results.append({
                    'Symbol': cik_to_symbol.get(CIK, 'Unknown'),
                    'CIK': CIK,
                    'Year': year,
                    'Filing Date': filing_date,
                    'Cosine Distance': cos_distance
                })

        previous_vector = current_vector
        previous_filing_date = filing_date

# --- Step 5: Create a DataFrame ---
results_df = pd.DataFrame(final_results)

# Format Filing Date into a datetime
results_df['Filing Date'] = pd.to_datetime(results_df['Filing Date'], format='%Y%m%d')

# Create a new column: Filing Month
results_df['Filing Month'] = results_df['Filing Date'].dt.month

# Reformat Filing Date back to just YYYY-MM-DD
results_df['Filing Date'] = results_df['Filing Date'].dt.date

# Optional: format Filing Date into YYYY-MM-DD
results_df['Filing Date'] = pd.to_datetime(results_df['Filing Date'], format='%Y%m%d').dt.date

# --- Step 6: Display or save the final DataFrame ---
print("\n--- Final DataFrame Preview ---")
print(results_df.head())

# Save it to CSV
output_path = os.path.join(home, 'Downloads', 'sp500_cosine_distances.csv')
results_df.to_csv(output_path, index=False)
print(f"\nSaved results to: {output_path}")

# --- Step 7: End timer ---
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal processing time: {elapsed_time:.2f} seconds")





--- Final DataFrame Preview ---
  Symbol    CIK  Year Filing Date  Cosine Distance  Filing Month
0    MMM  66740  2004  2005-02-24         0.004749             2
1    MMM  66740  2005  2006-02-21         0.010077             2
2    MMM  66740  2006  2007-02-26         0.004970             2
3    MMM  66740  2007  2008-02-15         0.006057             2
4    MMM  66740  2008  2009-02-17         0.007102             2

Saved results to: /Users/akankshagavade/Downloads/sp500_cosine_distances.csv

Total processing time: 45.68 seconds


In [2]:
permno_map = pd.read_csv('inputs/permno_cik_map.csv')
results_df['CIK'] = results_df['CIK'].astype(str).str.zfill(10)
permno_map['CIK'] = permno_map['cik'].astype(str).str.zfill(10)
results_df = results_df.merge(permno_map, on='CIK', how='left')

results_df

,Symbol,CIK,Year,Filing Date,Cosine Distance,Filing Month,tic,cik,permno
0,MMM,0000066740,2004,2005-02-24,0.004749,2,mmm,66740.0,22592.0
1,MMM,0000066740,2005,2006-02-21,0.010077,2,mmm,66740.0,22592.0
2,MMM,0000066740,2006,2007-02-26,0.004970,2,mmm,66740.0,22592.0
3,MMM,0000066740,2007,2008-02-15,0.006057,2,mmm,66740.0,22592.0
4,MMM,0000066740,2008,2009-02-17,0.007102,2,mmm,66740.0,22592.0
...,...,...,...,...,...,...,...,...,...
12118,ZTS,0001555280,2019,2020-02-13,0.002250,2,zts,1555280.0,13788.0
12119,ZTS,0001555280,2020,2021-02-16,0.001143,2,zts,1555280.0,13788.0
12120,ZTS,0001555280,2021,2022-02-15,0.000587,2,zts,1555280.0,13788.0
12121,ZTS,0001555280,2022,2023-02-14,0.020338,2,zts,1555280.0,13788.0


In [5]:
crsp_monthly = pd.read_csv('inputs/crsp_data.csv')

crsp_monthly['date'] = pd.to_datetime(crsp_monthly['date'])
crsp_monthly['filing_month'] = crsp_monthly['date'].dt.month
crsp_monthly['filing_YEAR'] = crsp_monthly['date'].dt.year

crsp_monthly

,permno,date,ret,filing_month,filing_YEAR
0,10000,1986-01-31,NaN,1,1986
1,10000,1986-02-28,-25.7143,2,1986
2,10000,1986-03-31,36.5385,3,1986
3,10000,1986-04-30,-9.8592,4,1986
4,10000,1986-05-30,-22.2656,5,1986
...,...,...,...,...,...
4047625,93436,2024-08-30,-7.7391,8,2024
4047626,93436,2024-09-30,22.1942,9,2024
4047627,93436,2024-10-31,-4.5025,10,2024
4047628,93436,2024-11-29,38.1469,11,2024


In [10]:
results_df['Year'] = pd.to_numeric(results_df['Year']).astype('Int64')

merged_df = pd.merge(results_df, crsp_monthly,
                     left_on=['Filing Month', 'Year', 'permno'],
                     right_on=['filing_month', 'filing_YEAR', 'permno'],
                     how='left')
merged_df


,Symbol,CIK,Year,Filing Date,Cosine Distance,Filing Month,tic,cik,permno,date,ret,filing_month,filing_YEAR
0,MMM,0000066740,2004,2005-02-24,0.004749,2,mmm,66740.0,22592.0,2004-02-27,-0.8977,2.0,2004.0
1,MMM,0000066740,2005,2006-02-21,0.010077,2,mmm,66740.0,22592.0,2005-02-28,0.0000,2.0,2005.0
2,MMM,0000066740,2006,2007-02-26,0.004970,2,mmm,66740.0,22592.0,2006-02-28,1.7869,2.0,2006.0
3,MMM,0000066740,2007,2008-02-15,0.006057,2,mmm,66740.0,22592.0,2007-02-28,0.3499,2.0,2007.0
4,MMM,0000066740,2008,2009-02-17,0.007102,2,mmm,66740.0,22592.0,2008-02-29,-0.9416,2.0,2008.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12118,ZTS,0001555280,2019,2020-02-13,0.002250,2,zts,1555280.0,13788.0,2019-02-28,9.3663,2.0,2019.0
12119,ZTS,0001555280,2020,2021-02-16,0.001143,2,zts,1555280.0,13788.0,2020-02-28,-0.7302,2.0,2020.0
12120,ZTS,0001555280,2021,2022-02-15,0.000587,2,zts,1555280.0,13788.0,2021-02-26,0.6418,2.0,2021.0
12121,ZTS,0001555280,2022,2023-02-14,0.020338,2,zts,1555280.0,13788.0,2022-02-28,-3.0732,2.0,2022.0


In [4]:
crsp_monthly

,permno,date,ret
0,10000,1986-01-31,NaN
1,10000,1986-02-28,-25.7143
2,10000,1986-03-31,36.5385
3,10000,1986-04-30,-9.8592
4,10000,1986-05-30,-22.2656
...,...,...,...
4047625,93436,2024-08-30,-7.7391
4047626,93436,2024-09-30,22.1942
4047627,93436,2024-10-31,-4.5025
4047628,93436,2024-11-29,38.1469
